In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch04-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "04-03-lipinski-rules.py"

# 四項警示與 BACE 活性

第一次執行需連網下載 BACE。要做小練習，可將自己的程式接到寫入分析程式的儲存格末端，重新執行該格及最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import deepchem as dc
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# 保留原始列次。描述符、門檻及製圖均無隨機抽樣。
tasks, (data,), _ = dc.molnet.load_bace_regression(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
assert tasks == ["pIC50"]
rows = []
for row, (smiles, y, w) in enumerate(zip(data.ids, data.y[:, 0], data.w[:, 0])):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or not w or not np.isfinite(y):
        continue
    if len(Chem.GetMolFrags(mol)) != 1 or any(a.GetFormalCharge() for a in mol.GetAtoms()):
        continue
    rows.append((row, Chem.MolToSmiles(mol), Descriptors.MolWt(mol),
                 Crippen.MolLogP(mol), Lipinski.NumHDonors(mol),
                 Lipinski.NumHAcceptors(mol), y))
table = pd.DataFrame(rows, columns=["row", "SMILES", "MW", "cLogP", "HBD", "HBA", "pIC50"])
limits = pd.Series({"MW": 500.0, "cLogP": 5.0, "HBD": 5, "HBA": 10})
flags = table[limits.index].gt(limits, axis="columns")
table["alerts"] = flags.sum(axis=1)
print("source rows / eligible rows:", len(data), len(table))
print("flags by descriptor:", flags.sum().to_dict())
print(table.groupby("alerts").pIC50.agg(["size","min","median","max"]).round(3))
print("examples (first eligible row at each alert count)")
print(table.groupby("alerts", sort=True).head(1).round(3).to_string(index=False))
counts = table.alerts.value_counts().reindex(range(5), fill_value=0)
fig, (left, right) = plt.subplots(1, 2, figsize=(9, 4), layout="constrained")
left.bar(counts.index, counts, color="#227872")
left.set(xlabel="Number of Ro5 alerts", ylabel="BACE records", xticks=range(5))
left.bar_label(left.containers[0])
groups = sorted(table.alerts.unique())
right.boxplot([table.loc[table.alerts == i, "pIC50"] for i in groups],
              positions=groups, widths=.55)
right.set(xlabel="Number of Ro5 alerts", ylabel="Measured BACE-1 pIC50", xticks=groups)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.savefig(out/"04-03-lipinski-rules.png", dpi=160)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "04-03-lipinski-rules.png")))